# 03 — MCP Resources

Resources are **application-controlled** primitives — your code decides when to fetch data.

## Learning Objectives

- Understand the difference between Resources and Tools
- Define static and templated resources
- Work with URI patterns and MIME types
- Read resources from a client

## Resources vs. Tools

| Aspect | Resources | Tools |
|--------|-----------|-------|
| Controlled by | Application code | Model (Claude) |
| Purpose | Read data | Perform actions |
| When used | App explicitly fetches | Claude decides |
| Direction | Data retrieval only | Action execution |

**Rule of thumb**: Reading data to display or augment context → Resource. Claude needs to perform an action → Tool.

In [ ]:
# Our document store
docs = {
    "report.pdf": "The report details the state of a 20m condenser tower.",
    "plan.md": "The plan outlines the steps for implementation.",
    "spec.txt": "Technical requirements for the equipment.",
    "financials.docx": "Budget and expenditure details for Q3.",
}

print("Document store ready with", len(docs), "documents")

## Static Resources

A fixed URI that always returns the same type of data:

In [ ]:
# Static resource: list all document IDs
# In a real MCP server, this would be:
# @mcp.resource("docs://documents", mime_type="application/json")

def list_documents() -> list[str]:
    """List all available document IDs."""
    return list(docs.keys())

# Simulate the resource call
import json

result = list_documents()
print("URI: docs://documents")
print("MIME: application/json")
print("Response:", json.dumps(result, indent=2))

## Templated Resources

Parameterized URIs with placeholders that become function arguments:

In [ ]:
# Templated resource: fetch a specific document
# In a real MCP server, this would be:
# @mcp.resource("docs://documents/{doc_id}", mime_type="text/plain")

def fetch_document(doc_id: str) -> str:
    """Fetch the contents of a specific document."""
    if doc_id not in docs:
        raise ValueError(f"Doc with id '{doc_id}' not found")
    return docs[doc_id]

# Simulate different URI calls
for doc_id in ["report.pdf", "plan.md"]:
    print(f"URI: docs://documents/{doc_id}")
    print(f"MIME: text/plain")
    print(f"Response: {fetch_document(doc_id)}")
    print()

## MIME Types

MIME types tell the client how to parse the response:

| MIME Type | Use Case | Client Handling |
|-----------|----------|----------------|
| `application/json` | Structured data | `json.loads(text)` |
| `text/plain` | Raw text | Use as-is |
| `text/markdown` | Markdown docs | Use as-is |
| `text/html` | HTML content | Use as-is |

In [ ]:
# Client-side resource parsing based on MIME type

def parse_resource(text: str, mime_type: str):
    """Parse a resource response based on its MIME type."""
    if mime_type == "application/json":
        return json.loads(text)
    return text

# JSON resource
json_text = json.dumps(["report.pdf", "plan.md", "spec.txt"])
parsed = parse_resource(json_text, "application/json")
print(f"JSON parsed: {parsed} (type: {type(parsed).__name__})")

# Plain text resource  
plain_text = "The report details the state of a 20m condenser tower."
parsed = parse_resource(plain_text, "text/plain")
print(f"Plain text: {parsed} (type: {type(parsed).__name__})")

## Complete Resource Server Code

Here's how resources look in a real MCP server:

In [ ]:
RESOURCE_SERVER_CODE = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("DocumentMCP", log_level="ERROR")

docs = {
    "report.pdf": "The report details the state of a 20m condenser tower.",
    "plan.md": "The plan outlines the steps for implementation.",
}

# Static resource — list all documents
@mcp.resource("docs://documents", mime_type="application/json")
def list_docs() -> list[str]:
    return list(docs.keys())

# Templated resource — fetch specific document  
@mcp.resource("docs://documents/{doc_id}", mime_type="text/plain")
def fetch_doc(doc_id: str) -> str:
    if doc_id not in docs:
        raise ValueError(f"Doc with id {doc_id} not found")
    return docs[doc_id]

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

print("Key patterns:")
print("  @mcp.resource(uri, mime_type=...) — registers the resource")
print("  Static:    'docs://documents'           — no parameters")
print("  Templated: 'docs://documents/{doc_id}'  — doc_id → function arg")
print("  Return values are auto-serialized to strings by the SDK")

## Exercise: Create Custom Resources

1. Create a static resource that returns server status info
2. Create a templated resource for searching documents by keyword

In [ ]:
# Exercise 1: Server status resource
# URI: status://health
# MIME: application/json

def server_status() -> dict:
    return {
        "status": "healthy",
        "document_count": len(docs),
        "documents": list(docs.keys()),
    }

print("Status resource:")
print(json.dumps(server_status(), indent=2))

In [ ]:
# Exercise 2: Search resource
# URI: docs://search/{query}
# MIME: application/json

def search_docs(query: str) -> list[str]:
    """Return doc IDs that contain the query string."""
    return [
        doc_id for doc_id, content in docs.items()
        if query.lower() in content.lower()
    ]

print("Search for 'report':")
print(json.dumps(search_docs("report"), indent=2))
print()
print("Search for 'steps':")
print(json.dumps(search_docs("steps"), indent=2))